# 2 · Explore the released encodings

Beyond the encoder itself, EncoTESS bundles the **PCA encoding** and a **2-D UMAP
embedding** for every released light curve, so you can explore the latent space without
re-encoding anything. This notebook loads those bundled arrays and:

1. inspects what's in them,
2. plots the UMAP colored a couple of ways,
3. finds a light curve's nearest neighbours in the encoding.

Plotting uses `matplotlib` (`pip install encotess[examples]`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from encotess import assets

pca = np.load(assets.pca_preview_path(), allow_pickle=False)   # top-64 PCA, all light curves
umap = np.load(assets.umap_path(), allow_pickle=False)         # 2-D UMAP, all light curves

Z = pca['latents_pca64']       # (N, 64) PCA encoding
emb = umap['embedding']        # (N, 2) UMAP coordinates
sectors = pca['sectors']       # TESS sector per light curve
bank = pca['bank']             # which released sub-sample each light curve belongs to
gaia = pca['gaia_ids']         # Gaia DR3 source id (string)
tic = pca['tic_ids']           # TIC id

print('PCA encoding :', Z.shape)
print('UMAP embedding:', emb.shape)
print('light curves  :', len(Z), ' | unique stars:', len(np.unique(gaia)))
print('sub-samples   :', {b: int((bank == b).sum()) for b in np.unique(bank)})

Each row is one **(star, sector)** light curve; a star observed in several sectors appears
several times. The bundled PCA (`latents_pca64.npz`) and UMAP (`umap.npz`) are row-aligned
to the same identifiers, so any coloring you compute from one applies to the other.

In [ ]:
# The two bundles line up row-for-row.
assert np.array_equal(pca['tic_ids'], umap['tic_ids'])
assert np.array_equal(pca['sectors'], umap['sectors'])
print('pca64 and umap bundles are row-aligned  ✓')

### UMAP colored by sub-sample

The released light curves come from three sub-samples; coloring the UMAP by that label
shows how they occupy the latent space.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for b in np.unique(bank):
    m = bank == b
    ax.scatter(emb[m, 0], emb[m, 1], s=2, alpha=0.35, label=f'{b} (n={m.sum()})')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.set_title('EncoTESS latent space (UMAP), colored by sub-sample')
ax.legend(markerscale=4, framealpha=0.9)
plt.tight_layout(); plt.show()

### UMAP colored by TESS sector

Coloring by observing sector is a useful diagnostic — structure that tracks the sector
would hint that instrument/observing effects leak into the encoding. (A handful of rows
carry out-of-range sector values, so we clip the color scale to the normal 1–90 range.)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
sc = ax.scatter(emb[:, 0], emb[:, 1], c=sectors, s=2, alpha=0.35,
                cmap='viridis', vmin=1, vmax=90)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.set_title('UMAP colored by TESS sector')
plt.colorbar(sc, ax=ax, label='sector (clipped to 1–90)')
plt.tight_layout(); plt.show()

### Nearest neighbours in the encoding

Distances in the PCA encoding are a simple way to find light curves the model considers
similar. Here we pick one light curve and rank the rest by Euclidean distance in the
top-64 PCA space.

In [ ]:
q = 0                                        # query = the first light curve
d2 = ((Z - Z[q]) ** 2).sum(axis=1)
order = np.argsort(d2)[:6]                   # itself + 5 nearest

print(f'query: gaia {gaia[q]}  (TIC {tic[q]}, sector {sectors[q]})\n')
print('nearest neighbours in PCA-64 space:')
for i in order:
    tag = '  <- query' if i == q else ''
    print(f'  gaia {gaia[i]:>20}  TIC {tic[i]:>10}  sector {sectors[i]:>3}  '
          f'dist={np.sqrt(d2[i]):7.2f}{tag}')

### Going further

- **Full-resolution encoding.** The bundled file is the top-64 PCA (~98% of the variance).
  The complete 1536-component PCA — a lossless re-expression of every latent — is available
  as an on-demand download via `assets.download_latents_pca()`.
- **Richer metadata.** Per-light-curve and per-star metadata tables (magnitudes, colors,
  parallax, …) are available via `assets.download_metadata('per_sector')` and
  `assets.download_metadata('per_star')`, keyed by the same identifiers used here.